# 03 · Motivation (c) — Regime-switching thresholds

**Toy example.** Each agent switches between a mild and a strong response regime
at its own threshold ξᵢ:

$$ a_i(x) = \begin{cases} -k_1 (x-\xi_i) & x < \xi_i \\ -k_2 (x-\xi_i) & x \ge \xi_i \end{cases} $$

A global cubic polynomial fitted to the pooled data has a *single* switching point
and therefore cannot represent the spread of thresholds across agents. We quantify
this with a Threshold Spread Index (TSI = max ξ − min ξ) recovered by the
personalized model versus the truth.

**Figures produced:** `regime_switch.png`, `regime_switch_predictions.png`,
`regime_switch_common.png`, `regime_switch_unique.png`, `regime_switch_tsi.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA

from bhd import FIG_DIR, set_style

set_style()

## 1. Agent responses with fixed thresholds, and the global polynomial model

In [ ]:
n_agents = 5
n_samples_per_agent = 100
x_range = np.linspace(0, 2, n_samples_per_agent)
k1, k2 = 0.5, 2.0
fixed_thresholds = np.linspace(0.6, 1.4, n_agents)

X_all, y_all, true_curves = [], [], []
for xi in fixed_thresholds:
    x = x_range.copy()
    a = np.where(x < xi, -k1 * (x - xi), -k2 * (x - xi))
    X_all.append(x)
    y_all.append(a)
    true_curves.append((x, a, xi))

X_all_concat = np.concatenate(X_all).reshape(-1, 1)
y_all_concat = np.concatenate(y_all)

poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X_all_concat)
global_model = LinearRegression().fit(X_poly, y_all_concat)
x_plot = np.linspace(0, 2, 200).reshape(-1, 1)
y_global = global_model.predict(poly.transform(x_plot))

## 2. Two-shot decomposition

In [ ]:
X_matrix = np.array(X_all).T  # (100, 5)
Y_matrix = np.array(y_all).T  # (100, 5)

pca_shared = PCA(n_components=1)
C = pca_shared.fit(Y_matrix.T).components_.T
Psi = C.T @ Y_matrix
Y_shared = C @ Psi

personalized, reconstructed = [], []
for v in range(n_agents):
    R_v = (Y_matrix[:, v] - Y_shared[:, v]).reshape(-1, 1)
    pca_personal = PCA(n_components=1)
    Phi_v = pca_personal.fit_transform(R_v)
    R_hat = pca_personal.inverse_transform(Phi_v)
    personalized.append(R_hat.flatten())
    reconstructed.append(Y_shared[:, v] + R_hat.flatten())

Y_pred_personalized = np.column_stack(reconstructed)
Y_shared_matrix = Y_shared
Y_personal_matrix = np.column_stack(personalized)

## 3. Threshold estimation and the Threshold Spread Index (TSI)

In [ ]:
def estimate_switching_threshold(x, y):
    """Location of the largest absolute slope change."""
    dy = np.gradient(y, x)
    return x[np.argmax(np.abs(dy))]


true_thresholds = fixed_thresholds
personalized_thresholds = [
    estimate_switching_threshold(X_matrix[:, 0], Y_pred_personalized[:, i]) for i in range(n_agents)
]
global_threshold = estimate_switching_threshold(x_plot.ravel(), y_global)

tsi_true = np.max(true_thresholds) - np.min(true_thresholds)
tsi_pers = np.max(personalized_thresholds) - np.min(personalized_thresholds)
print(f"True thresholds:         {np.round(true_thresholds, 3)}")
print(f"Personalized thresholds: {np.round(personalized_thresholds, 3)}")
print(f"Global threshold:        {global_threshold:.3f}")
print(f"TSI (true) = {tsi_true:.3f}   TSI (personalized) = {tsi_pers:.3f}")

## 4. Figures

In [ ]:
colors = cm.viridis(np.linspace(0, 1, n_agents))

# True behavior and global model
plt.figure(figsize=(10, 7))
for i, (x, a, _) in enumerate(true_curves):
    plt.plot(x, a, color=colors[i], alpha=0.25, linewidth=7)
plt.plot([], [], color="black", linewidth=7, alpha=0.25, label="Vehicle i")
plt.plot(x_plot, y_global, label="Global Model", linestyle="-", color="black", linewidth=4)
plt.xlabel("Stimulus $x$", fontsize=22)
plt.ylabel("Acceleration $a(x)$", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.grid(True)
plt.legend(fontsize=22, loc="lower center", ncol=2)
plt.tight_layout()
plt.savefig(FIG_DIR / "regime_switch.png", dpi=300, bbox_inches="tight")
plt.show()

# Personalized predictions
plt.figure(figsize=(10, 7))
for i, (x, a, _) in enumerate(true_curves):
    plt.plot(x, a, color=colors[i], alpha=0.25, linewidth=7)
    plt.plot(x, Y_pred_personalized[:, i], linestyle=":", color=colors[i], linewidth=4)
plt.xlabel("Stimulus $x$", fontsize=22)
plt.ylabel("Acceleration $a(x)$", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.grid(True)
plt.legend(["Vehicle i", "Personalized Prediction i"], fontsize=22, loc="lower center", ncol=3)
plt.tight_layout()
plt.savefig(FIG_DIR / "regime_switch_predictions.png", dpi=300, bbox_inches="tight")
plt.show()

# Shared component
plt.figure(figsize=(10, 7))
for i in range(n_agents):
    plt.plot(X_matrix[:, 0], Y_shared_matrix[:, i], color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("Stimulus $x$", fontsize=22)
plt.ylabel("Acceleration $a(x)$", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.ylim(-1.9, 0.8)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "regime_switch_common.png", dpi=300, bbox_inches="tight")
plt.show()

# Personalized component
plt.figure(figsize=(10, 7))
for i in range(n_agents):
    plt.plot(X_matrix[:, 0], Y_personal_matrix[:, i], linestyle="--", color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("Stimulus $x$", fontsize=22)
plt.ylabel("Acceleration $a(x)$", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.ylim(-1.9, 0.8)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "regime_switch_unique.png", dpi=300, bbox_inches="tight")
plt.show()

# Threshold Spread Index
plt.figure(figsize=(10, 7))
plt.bar(["True Thresholds", "Personalized Model"], [tsi_true, tsi_pers], color=["red", "blue"], alpha=0.9)
plt.axhline(global_threshold, color="black", linestyle="-", linewidth=2, label="Global Model")
plt.ylabel("Threshold Spread Index (TSI)", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.grid(True)
plt.legend(fontsize=22)
plt.tight_layout()
plt.savefig(FIG_DIR / "regime_switch_tsi.png", dpi=300, bbox_inches="tight")
plt.show()